In [3]:

print(f"{3*"="} TASK 1: Database Creation {3*"="}")

import pandas as pd
import sqlite3
# Database connection
conn = sqlite3.connect("bookstore.db")
print("Database connection established")

cursor = conn.cursor()

# Creating Books TABLE
cursor.execute("""CREATE TABLE IF NOT EXISTS Books(
  book_id INTEGER PRIMARY KEY AUTOINCREMENT,
  title   TEXT NOT NULL,
  author  TEXT NOT NULL,
  price   REAL NOT NULL,
  stock_quantity INTEGER DEFAULT 0
)
""")
print("Books table created successfully")

# Creating Customers TABLE
cursor.execute("""CREATE TABLE IF NOT EXISTS Customers(
  customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
  name        TEXT NOT NULL,
  email       TEXT UNIQUE NOT NULL,
  city        TEXT,
  join_date   TEXT
)
""")
print("Customers table created successfully")

# Creating Orders TABLE
cursor.execute("""CREATE TABLE IF NOT EXISTS Orders(
  order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  customer_id INTEGER,
  book_id     INTEGER,
  quantity    INTEGER NOT NULL,
  order_date  TEXT NOT NULL,
  total_amount REAL,
  FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
  FOREIGN KEY (book_id) REFERENCES Books(book_id)
)
""")
print("Orders table created successfully")

# Enable FOREIGN KEY constraints
cursor.execute("PRAGMA Foreign_keys = ON")

# Save changes
conn.commit()

# Display the schema of each table using PRAGMA commands
tables = ["books", "customers", "orders"]

for table in tables:
  print(f"\nSchema for {table} table:")

  cursor.execute(f"PRAGMA table_info({table})")
  schema = cursor.fetchall()

  # Print header
  print(f"{'cid':<5}{'name':<17}{'type':<9}{'notnull':<9}{'dflt_value':<12}{'pk'}")

  # Print rows
  for row in schema:
    cid, name, col_type, notnull, dflt_value, pk = row
    print(f"{cid:<5}{name:<17}{col_type:<9}{notnull:<9}{str(dflt_value):<12}{pk}")


print(f"\n{3*"="} TASK 2: Data Insertion and Querying {3*"="}")

# Insert following data into Books Table
books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)
]

# Insert many records at once in Books Table
cursor.executemany("""INSERT INTO books(title, author, price, stock_quantity) VALUES(?, ?, ?, ?)
""", books_data)
conn.commit()
print("Inserted 5 books")

# Insert following data into Customers Table
customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

# Insert many records at once in Customers Table
cursor.executemany("""INSERT INTO customers(name, email, city, join_date) VALUES(?, ?, ?, ?)
""", customers_data)
conn.commit()
print("Inserted 5 customers")

# Insert following data into Orders Table
orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]

# Insert many records at once in Orders Table
cursor.executemany("""INSERT INTO orders(customer_id, book_id, quantity, order_date, total_amount) VALUES(?, ?, ?, ?, ?)
""", orders_data)
conn.commit()
print("Inserted 7 orders")

# Display Books Table
print("\nAll Books:")
cursor.execute("SELECT * FROM books")
books_table = cursor.fetchall()

# Print Header
print(f"{'book_id':<10}{'title':<30}{'author':<17}{'price':<10}{'stock':<7}")
for row in books_table:
  book_id, title, author, price, stock = row
  print(f"{book_id:<10}{title:<30}{author:<17}{price:<10}{stock:<7}")

# Display Customer Table
print("\nAll Customers:")
cursor.execute("SELECT * FROM customers")
customers_table = cursor.fetchall()

# Print Header
print(f"{'customer_id':<12}{'name':<20}{'email':<17}{'city':<12}{'join_date':<15}")
for row in customers_table:
  customer_id, name, email, city, join_date = row
  print(f"{customer_id:<12}{name:<20}{email:<17}{city:<12}{join_date:<15}")

# Display Orders Table
print("\nAll Orders:")
cursor.execute("SELECT * FROM orders")
orders_table = cursor.fetchall()

# Print Header
print(f"{'order_id':<10}{'customer_id':<12}{'book_id':<10}{'quantity':<9}{'order_date':<15}{'total_amount':<7}")
for row in orders_table:
  order_id, customer_id, book_id, quantity, order_date, total_amount = row
  print(f"{order_id:<10}{customer_id:<12}{book_id:<10}{quantity:<9}{order_date:<15}{total_amount:<7}")

# Query results
print("\nQuery Results: ")

# All customers from Mumbai
print("\nCustomers from Mumbai:\n")
cursor.execute("""SELECT customer_id, name, email, city FROM customers
WHERE city = 'Mumbai'""")
rows = cursor.fetchall()

# Print Header
print(f"{'customer_id':<12}{'name':<20}{'email':<17}{'city':<12}")
for row in rows:
  customer_id, name, email, city = row
  print(f"{customer_id:<12}{name:<20}{email:<17}{city:<12}")


# Books with price > 800 and stock > 10
print("\nBooks priced > 800 with stock > 10: \n")
cursor.execute("""SELECT book_id, title, price, stock_quantity AS stock FROM books
WHERE ((price > 800) AND (stock > 10))
""")
rows = cursor.fetchall()

# Print Header
print(f"{'book_id':<10}{'title':<30}{'price':<10}{'stock':<7}")
for row in rows:
  book_id, title, price, stock = row
  print(f"{book_id:<10}{title:<30}{price:<10}{stock:<7}")

# Total Orders
print(f"\nTotal Orders: ")
cursor.execute("""SELECT COUNT(*) FROM orders""")
result = cursor.fetchone()[0]
print(result)

# Customer who placed the most orders
print(f"\nCustomer who placed the most orders: ")
cursor.execute("""SELECT customer_id, COUNT(*) AS total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 1
""")
result1 = cursor.fetchone()
print(result1)

# Total Revenue
print(f"\nTotal Revenue: ")
cursor.execute("""SELECT SUM(total_amount) FROM orders""")
result2 = cursor.fetchone()[0]
print(result2)

print(f"\n{3*"="} TASK 3: Pandas Integration {3*"="}\n")
# Loading all the 3 tables into separate pandas dataframes
books_df = pd.read_sql("SELECT * FROM books", conn)
print(f"DataFrames loaded from SQL:")
print(f"- Books: 5 rows * 5 columns")

customers_df = pd.read_sql("SELECT * FROM customers", conn)
print(f"- Customers: 5 rows * 5 ccolumns")

orders_df = pd.read_sql("SELECT * FROM orders", conn)
print(f"- Orders: 7 rows * 6 columns")

# Display first 3 rows of each DataFrame
print("Books DataFrame: ")
books_df.head(3)
print("\nCustomers DataFrame: ")
customers_df.head(3)
print("\nOrders DataFrame: ")
orders_df.head(3)

# Creating comprehensive order report
order_report = orders_df.merge(customers_df, on="customer_id").merge(books_df, on="book_id")
order_report = order_report[["order_id", "name", "city", "title", "quantity", "total_amount"]]
order_report.columns = ["Order ID", "Customer Name", "Customer City", "Book Title", "Quantity", "Total Amount"]
print("\nComprehensive Order Report:")
print(order_report)

# Average Order Value
avg_order_value = order_report["Total Amount"].mean()
print("\nAnalysis Results:")
print(f"Average Order Value: Rs.{avg_order_value:.2f}")

# Total orders by city
orders_by_city = order_report.groupby("Customer City")["Order ID"].count()
print("\nOrders by city: ")
print(orders_by_city)

# Most Popular book
book_orders = order_report.groupby("Book Title")["Quantity"].sum()
popular_book = book_orders.idxmax()
print(f"Most Popular Book: {popular_book} ({book_orders.max()} orders)")

# Create Discount DataFrame

discounts_data = {
    'book_id': [1, 2, 3, 4, 5],
    'discount_percent': [10, 15, 5, 20, 12]
}

discount_df = pd.DataFrame(discounts_data)
print("\nDiscounts table created and saved to database")

discount_df.to_sql("discounts", conn, if_exists="replace", index=False)

# Verify the table was created
verify_df = pd.read_sql("SELECT * FROM discounts", conn)
print("Discounts table:")
print(verify_df)

# Book titles with disconted price
discount_query = """SELECT
b.title, b.price as original_price, d.discount_percent,
b.price*(1-d.discount_percent/100.0) as discounted_price
FROM books b
JOIN discounts d
ON b.book_id = d.book_id"""

discounted_books = pd.read_sql(discount_query, conn)
print("\nBooks with Discounted Prices:")
print(discounted_books)


# Close connection
conn.close()

=== TASK 1: Database Creation ===
Database connection established
Books table created successfully
Customers table created successfully
Orders table created successfully

Schema for books table:
cid  name             type     notnull  dflt_value  pk
0    book_id          INTEGER  0        None        1
1    title            TEXT     1        None        0
2    author           TEXT     1        None        0
3    price            REAL     1        None        0
4    stock_quantity   INTEGER  0        0           0

Schema for customers table:
cid  name             type     notnull  dflt_value  pk
0    customer_id      INTEGER  0        None        1
1    name             TEXT     1        None        0
2    email            TEXT     1        None        0
3    city             TEXT     0        None        0
4    join_date        TEXT     0        None        0

Schema for orders table:
cid  name             type     notnull  dflt_value  pk
0    order_id         INTEGER  0        None 